In [9]:
# import pandas
import pandas as pd

# import pyecharts
from pyecharts import options as opts
from pyecharts.charts import Bar, Line
from pyecharts import options as opts
from pyecharts.commons.utils import JsCode

In [10]:
#%%  Reading PKL file into a DataFrame

# Reading the pickle files into DataFrames
ethereum_tx = pd.read_pickle('files\\ethereum_tx.pkl')
bitcoin_tx = pd.read_pickle('files\\bitcoin_tx.pkl')
ripple_tx = pd.read_pickle('files\\ripple_tx.pkl')
bnb_smart_tx = pd.read_pickle('files\\bnb_smart_tx.pkl')
avalanche_tx = pd.read_pickle('files\\avalanche_tx.pkl')

# Read the Binance transaction data and assign it to bnb_beacon_tx
binance_tx = pd.read_pickle('files\\binance_tx.pkl')
bnb_beacon_tx = binance_tx
bnb_beacon_tx['chain'] = 'bnb_beacon'

In [11]:
# assume both have the same columns: ['chain','date','tx_count']
web3_tx = pd.concat(
    [ethereum_tx, bitcoin_tx, ripple_tx, bnb_smart_tx, bnb_beacon_tx, avalanche_tx],
    ignore_index=True
)

# Convert date to datetime and chain to categorical
web3_tx['chain'] = pd.Categorical(web3_tx['chain'])
web3_tx = web3_tx.sort_values(['chain','date']).reset_index(drop=True)

# Group by chain + week, summing tx_count
web3_tx = (
    web3_tx.groupby(['chain',pd.Grouper(key='date', freq='ME')])['tx_count']
    .sum()
    .reset_index(name='tx_count')
)

# Get the latest month and filter out rows from the last month
latest_month = web3_tx["date"].max()
web3_tx = web3_tx[web3_tx["date"] < latest_month].reset_index(drop=True)

# Exclude rows with a tx_count of 0
web3_tx = web3_tx[web3_tx['tx_count'] != 0]

# Calculate the total transactions per date and add it as a new column
web3_tx['total_tx'] = web3_tx.groupby('date')['tx_count'].transform('sum')

# Calculate the percentage for each tx_count relative to total_tx
web3_tx['pct_of_total'] = (web3_tx['tx_count'] / web3_tx['total_tx']) * 100

# Save and load the DataFrame
web3_tx = pd.DataFrame.to_pickle(web3_tx, 'files\\web3_tx.pkl')
web3_tx = pd.read_pickle('files\\web3_tx.pkl')

print(web3_tx.info())

<class 'pandas.core.frame.DataFrame'>
Index: 444 entries, 32 to 533
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   chain         444 non-null    category      
 1   date          444 non-null    datetime64[ns]
 2   tx_count      444 non-null    int64         
 3   total_tx      444 non-null    int64         
 4   pct_of_total  444 non-null    float64       
dtypes: category(1), datetime64[ns](1), float64(1), int64(2)
memory usage: 17.8 KB
None


C:\Users\Matheus\AppData\Local\Temp\ipykernel_29120\1715088580.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  web3_tx.groupby(['chain',pd.Grouper(key='date', freq='ME')])['tx_count']


In [12]:
from pyecharts.commons.utils import JsCode

# 1) Your data
dates = [...]      # list of datetime strings or x-axis labels
values = [...]     # corresponding large numbers

# 2) JS function to abbreviate numbers
abbr_formatter = JsCode("""
function (value) {
    if (value >= 1e12) { return +(value/1e12).toFixed(1) + 'T'; }
    if (value >= 1e9)  { return +(value/1e9).toFixed(1)  + 'B'; }
    if (value >= 1e6)  { return +(value/1e6).toFixed(1)  + 'M'; }
    if (value >= 1e3)  { return +(value/1e3).toFixed(1)  + 'K'; }
    return value;
}
""")

In [13]:
web3_tx = pd.read_pickle('files\\web3_tx.pkl')

# Ensure date column is in datetime, if needed
web3_tx['date'] = pd.to_datetime(web3_tx['date'])

# Get unique sorted dates as x-axis labels
dates = sorted(web3_tx['date'].unique())
x_axis = [d.strftime('%Y-%m-%d') for d in dates]

# Define the color mapping for chains
color_map = {
    'bitcoin': '#FC922F',   # Bitcoin Orange
    'ethereum': '#626AFF',  # Ethereum Blue
    'bnb_smart': '#FFCF3D',  # Binance Yellow
    'bnb_beacon': '#786834',  # Binance Yellow
    'ripple': '#DCDCDC',    # Ripple Grey
    'avalanche': '#FF3A3A',  # Avalanche Red
}

# Get unique chains and prepare data series for each
chains = web3_tx['chain'].unique()
data_series = {}

for chain in chains:
    # Filter rows for a given chain, using date as index
    df_chain = web3_tx[web3_tx['chain'] == chain].set_index('date')['tx_count']
    # Build a list of transaction counts aligned to the x_axis dates (fill missing as 0)
    series = [int(df_chain.get(d, 0)) for d in dates]
    data_series[chain] = series

# Create the stacked bar chart with pyecharts
bar = Bar(
    init_opts=opts.InitOpts(
        theme="dark",
        bg_color="rgba(0,0,0,0)",
        width="100%",
        height="535px",
    )
)

bar.add_xaxis(x_axis)

for chain, series in data_series.items():
    # Provide a custom item style if the chain exists in our color map
    itemstyle = None
    if chain in color_map:
        itemstyle = opts.ItemStyleOpts(color=color_map[chain])
    bar.add_yaxis(
        chain,
        series,
        stack="stack1",
        label_opts=opts.LabelOpts(is_show=False),
        itemstyle_opts=itemstyle,
    )

bar.set_global_opts(
    title_opts=opts.TitleOpts(title="Blockchain Monthly Transactions"),
    tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="shadow"),
    datazoom_opts=opts.DataZoomOpts(type_="slider", range_start=0, range_end=100),
    legend_opts=opts.LegendOpts(pos_left="right", is_show=False),
    yaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(formatter=abbr_formatter)),
    xaxis_opts=opts.AxisOpts(splitline_opts=opts.SplitLineOpts(is_show=False)),
)

bar.render('echarts\\blockchain_activity.html', template_name="simple_chart.html")
bar.render_notebook()  # For Jupyter Notebook display